In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# 1. Cargar los datos
df = pd.read_csv("../train (1).csv")

# 2. Limpieza básica (Los modelos de Machine Learning no toleran valores vacíos o NaN)
# Llenamos las edades vacías con la mediana de las edades
df['Age'] = df['Age'].fillna(df['Age'].median())
# Llenamos los puertos de embarque vacíos con el más común
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# 3. Convertir texto a números
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

# Seleccionamos las características que usaremos para predecir
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']
X = df[features]
y = df['Survived'] # Esta es nuestra variable objetivo (lo que queremos predecir)

print("Datos preparados y listos.")

: 

In [ ]:
# Para lograr un 70/15/15, lo hacemos en dos pasos:

# Paso A: Separar el 70% para entrenamiento y el 30% restante para (validación + prueba)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)

# Paso B: Del 30% restante, separamos la mitad (50%) para validación y la otra mitad para prueba. 
# 50% de 30% = 15% del total original.
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

print(f"Total de datos: {len(X)}")
print(f"Datos de Entrenamiento (70%): {len(X_train)}")
print(f"Datos de Evaluación/Validación (15%): {len(X_val)}")
print(f"Datos de Prueba/Test (15%): {len(X_test)}")

In [ ]:
columnas_a_escalar = ['Age', 'Fare', 'SibSp']

# --- A. Sin Escalar ---
# Ya los tenemos en X_train, X_val, X_test

# --- B. StandardScaler ---
scaler_std = StandardScaler()
X_train_std = X_train.copy()
X_val_std = X_val.copy()
X_test_std = X_test.copy()

X_train_std[columnas_a_escalar] = scaler_std.fit_transform(X_train[columnas_a_escalar])
X_val_std[columnas_a_escalar] = scaler_std.transform(X_val[columnas_a_escalar])
X_test_std[columnas_a_escalar] = scaler_std.transform(X_test[columnas_a_escalar])

# --- C. MinMaxScaler ---
scaler_minmax = MinMaxScaler()
X_train_mm = X_train.copy()
X_val_mm = X_val.copy()
X_test_mm = X_test.copy()

X_train_mm[columnas_a_escalar] = scaler_minmax.fit_transform(X_train[columnas_a_escalar])
X_val_mm[columnas_a_escalar] = scaler_minmax.transform(X_val[columnas_a_escalar])
X_test_mm[columnas_a_escalar] = scaler_minmax.transform(X_test[columnas_a_escalar])

print("Datos escalados con éxito en sus respectivas versiones.")

In [ ]:
# Definimos una función para entrenar y evaluar fácilmente y no repetir código
def evaluar_modelo(modelo, X_t, y_t, X_v, y_v, nombre_datos):
    modelo.fit(X_t, y_t) # El modelo aprende con los datos de entrenamiento
    predicciones = modelo.predict(X_v) # El modelo se pone a prueba con validación
    precision = accuracy_score(y_v, predicciones)
    print(f"Precisión con {nombre_datos}: {precision * 100:.2f}%")

# 1. Instanciamos los modelos
# Fijamos random_state en el Árbol para que los resultados sean consistentes
arbol = DecisionTreeClassifier(random_state=42) 
knn = KNeighborsClassifier(n_neighbors=5)

print("--- RESULTADOS ÁRBOL DE DECISIÓN ---")
# Teoría: Deberían dar resultados idénticos o casi idénticos, porque al árbol no le afectan las escalas.
evaluar_modelo(arbol, X_train, y_train, X_val, y_val, "Datos Originales")
evaluar_modelo(arbol, X_train_std, y_train, X_val_std, y_val, "StandardScaler")
evaluar_modelo(arbol, X_train_mm, y_train, X_val_mm, y_val, "MinMaxScaler")

print("\n--- RESULTADOS K-NEIGHBORS (KNN) ---")
# Teoría: Los datos escalados deberían tener un mejor rendimiento que los originales.
evaluar_modelo(knn, X_train, y_train, X_val, y_val, "Datos Originales")
evaluar_modelo(knn, X_train_std, y_train, X_val_std, y_val, "StandardScaler")
evaluar_modelo(knn, X_train_mm, y_train, X_val_mm, y_val, "MinMaxScaler")